# Fraud Detection Training

Train Isolation Forest on transaction-level features for anti-gaming detection.

In [11]:
import pandas as pd
df = pd.read_parquet("../data/synth_transactions.parquet")

# Only consider inflows for fraud injection (we inject fraudulent incoming txns)
inflows = df[df["type"] == "inflow"]

user_stats = inflows.groupby("user_id").agg(
    n_txns=("amount_kobo", "count"),
    span_days=("occurred_at", lambda x: (x.max() - x.min()).days),
    median_amount=("amount_kobo", "median"),
).reset_index()

eligible = user_stats[
    (user_stats.n_txns >= 20) & 
    (user_stats.span_days >= 14)
]

print(f"Total users: {user_stats.shape[0]}")
print(f"Eligible users (>=20 inflows, >=14 day span): {eligible.shape[0]}")
print(f"\nEligible user median-amount distribution (kobo):")
print(eligible.median_amount.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
print(f"\nEligible user n_txns distribution:")
print(eligible.n_txns.describe(percentiles=[0.1, 0.5, 0.9]))

Total users: 10000
Eligible users (>=20 inflows, >=14 day span): 10000

Eligible user median-amount distribution (kobo):
count    1.000000e+04
mean     2.960120e+05
std      3.044209e+05
min      2.309000e+04
10%      4.926960e+04
25%      6.791225e+04
50%      1.918305e+05
75%      3.434062e+05
90%      8.695864e+05
max      1.648519e+06
Name: median_amount, dtype: float64

Eligible user n_txns distribution:
count    10000.000000
mean      2268.617400
std       2147.285894
min        306.000000
10%        537.000000
50%       1356.000000
90%       6556.100000
max       8034.000000
Name: n_txns, dtype: float64


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
import pickle
from pathlib import Path

# Import our modules
import sys
sys.path.append('../')
from training.synthetic_data_gen import generate_synthetic_data
from training.fraud_feature_engine import compute_transaction_features

## Generate Synthetic Transaction Data

In [ ]:
# Generate synthetic data for multiple users
np.random.seed(42)
n_users = 1000
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 1, 1)

all_txns = []
for user_id in range(n_users):
    user_txns = generate_synthetic_data(
        user_id=str(user_id),
        start_date=start_date,
        end_date=end_date,
        archetype=None  # random archetype
    )
    all_txns.append(user_txns)

transaction_df = pd.concat(all_txns, ignore_index=True)
print(f"Generated {len(transaction_df)} transactions for {n_users} users")
transaction_df.head()

## Compute Transaction-Level Features

In [ ]:
# Compute features for all transactions
# For training, we use the end_date as as_of
as_of = end_date

# Group by user and compute features
feature_dfs = []
for user_id, user_txns in transaction_df.groupby('user_id'):
    user_features = compute_transaction_features(user_txns, as_of)
    if len(user_features) > 0:
        feature_dfs.append(user_features)

all_features = pd.concat(feature_dfs, ignore_index=True)
print(f"Computed features for {len(all_features)} transactions")
all_features.head()

## Train Isolation Forest Model

In [ ]:
from inference.fraud_predictor import FraudPredictor

# Train the model
contamination = 0.05  # assume 5% fraud
fraud_predictor = FraudPredictor.train(all_features, contamination=contamination)

print("Isolation Forest trained successfully")

## Compute Anomaly Scores

In [ ]:
# Predict anomaly scores
features_with_scores = fraud_predictor.predict_transaction_anomalies(all_features)
print(f"Anomaly scores computed. Mean: {features_with_scores['anomaly_score'].mean():.3f}, Max: {features_with_scores['anomaly_score'].max():.3f}")
features_with_scores[['anomaly_score']].describe()

## Save the Trained Model

In [ ]:
# Load existing artifact and add fraud model
artifact_path = Path('../models/fraud_model_artifact_v1.pkl')
with open(artifact_path, 'rb') as f:
    artifact = pickle.load(f)

# Add fraud model
artifact['fraud_model'] = fraud_predictor.model

# Save updated artifact
with open(artifact_path, 'wb') as f:
    pickle.dump(artifact, f)

print("Fraud model added to artifact and saved")